In [1]:
import pandas as pd
import yfinance as yf
from itertools import product
import time
import json
from datetime import datetime
import duckdb
from dotenv import load_dotenv
import os
import requests
import s3fs
import duckdb 
import polars as pl
import boto3
load_dotenv()



True

In [2]:

s3 = s3fs.S3FileSystem(
    key=os.getenv("S3_ACCESS_KEY"),
    secret=os.getenv("S3_SECRET_KEY"),
    token=os.getenv("AWS_SESSION_TOKEN"),  # optional but important if using temp creds
    client_kwargs={
        "region_name": "ca-central-1"
    }
)

In [3]:
# scan s3 bucket using duckdb

con = duckdb.connect()

bucket = os.getenv("S3_BUCKET")

snp500 = con.execute(f"""
SELECT *
FROM read_csv_auto('s3://{bucket}/post-earnings-forecast/**/*.csv')

""").df()
snp500

,#,Company,Symbol,Weight,Price,Chg,% Chg
0,1,NVIDIA Corp,NVDA,7.69%,215.33,-4.18,-1.90%
1,2,Apple Inc,AAPL,6.69%,308.82,3.83,1.26%
2,3,Microsoft Corp,MSFT,4.58%,418.57,-0.52,-0.12%
3,4,Amazon.com Inc,AMZN,4.22%,266.32,-2.14,-0.80%
4,5,Alphabet Inc,GOOGL,3.53%,382.97,-4.69,-1.21%
...,...,...,...,...,...,...,...
498,499,Pool Corp,POOL,0.01%,184.64,2.95,1.62%
499,500,Conagra Brands Inc,CAG,0.01%,13.56,0.18,1.35%
500,501,Campbell's Company/The,CPB,0.01%,20.58,0.53,2.64%
501,502,News Corp,NWS,0.01%,29.68,-0.40,-1.33%


In [4]:
# checking earnings data
q = f"""
SELECT *
FROM read_parquet(
  's3://{bucket}/post-earnings-forecast/earnings/**/*.parquet',
  hive_partitioning = true
);
"""
# get the data
earnings_df = con.execute(q).df()


In [10]:
earnings_df[earnings_df['symbol']=='MRNA'].reset_index(drop=True)

,fiscalDateEnding,reportedDate,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,symbol,fiscal_month,fiscal_year,av_quarter,time_from
0,2026-03-31,2026-05-01,-3.4,-2.11,-1.29,-61.1374,pre-market,MRNA,3,2026,2026Q1,20260501T0000
1,2025-12-31,2026-02-13,-2.11,-2.6,0.49,18.8462,pre-market,MRNA,12,2025,2025Q4,20260213T0000
2,2025-09-30,2025-11-06,-0.51,-2.16,1.65,76.3889,pre-market,MRNA,9,2025,2025Q3,20251106T0000
3,2025-06-30,2025-08-01,-2.13,-2.97,0.84,28.2828,pre-market,MRNA,6,2025,2025Q2,20250801T0000
4,2025-03-31,2025-05-01,-2.52,-3.11,0.59,18.9711,pre-market,MRNA,3,2025,2025Q1,20250501T0000
5,2024-12-31,2025-02-14,-2.91,-2.7599,-0.1501,-5.4386,pre-market,MRNA,12,2024,2024Q4,20250214T0000
6,2024-09-30,2024-11-07,0.03,-1.9,1.93,101.5789,pre-market,MRNA,9,2024,2024Q3,20241107T0000
7,2024-06-30,2024-08-01,-3.33,-3.36,0.03,0.8929,pre-market,MRNA,6,2024,2024Q2,20240801T0000
8,2024-03-31,2024-05-02,-3.07,-3.55,0.48,13.5211,pre-market,MRNA,3,2024,2024Q1,20240502T0000
9,2023-12-31,2024-02-22,0.55,-0.97,1.52,156.701,pre-market,MRNA,12,2023,2023Q4,20240222T0000


In [ ]:
# checking transcript data
q = f"""
SELECT *
FROM read_parquet(
  's3://{bucket}/post-earnings-forecast/transcripts/**/*.parquet',
  hive_partitioning = true
);
"""
# get the data
df = con.execute(q).df()
df['symbol'].value_counts()

In [24]:
paginator = s3.get_paginator("list_objects_v2")
symbols = []

for page in paginator.paginate(
    Bucket=bucket,
    Prefix="post-earnings-forecast/transcripts/",
    Delimiter="/"
):
    symbols += [
        p["Prefix"].split("symbol=")[1].rstrip("/")
        for p in page.get("CommonPrefixes", [])
        if "symbol=" in p["Prefix"]
    ]

print(len(symbols))

26


In [52]:
symbols

['AES',
 'AOS',
 'ARE',
 'BAX',
 'BLDR',
 'BXP',
 'CAG',
 'CPB',
 'CRL',
 'EPAM',
 'FDS',
 'FRT',
 'HSIC',
 'JKHY',
 'MGM',
 'MOS',
 'NCLH',
 'NVDA',
 'NWS',
 'NWSA',
 'POOL',
 'TAP',
 'TECH',
 'TTD',
 'UHS',
 'WYNN']

In [11]:
df = con.execute("""
    SELECT *
    FROM read_parquet('s3://docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=CHTR/**/*.parquet')
""").df()

In [12]:
df

,fiscalDateEnding,reportedDate,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,symbol,fiscal_month,fiscal_year,av_quarter,time_from,transcript
0,2013-12-31,2014-02-21,0.39,0.24,0.15,62.5,pre-market,CHTR,12,2013,2013Q4,20140221T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
1,2014-03-31,2014-04-28,-0.39,0.13,-0.52,-400,pre-market,CHTR,3,2014,2014Q1,20140428T0000,"[{'speaker': 'Tom Robey', 'title': 'SVP of Inv..."
2,2014-06-30,2014-07-31,-0.46,0.35,-0.81,-231.4286,pre-market,CHTR,6,2014,2014Q2,20140731T0000,"[{'speaker': 'Tom Robey', 'title': 'Senior Vic..."
3,2014-09-30,2014-10-29,-0.54,0.03,-0.57,-1900,pre-market,CHTR,9,2014,2014Q3,20141029T0000,"[{'speaker': 'Tom Robey', 'title': 'Senior Vic..."
4,2014-12-31,2015-02-05,-0.49,0.01,-0.5,-5000,pre-market,CHTR,12,2014,2014Q4,20150205T0000,"[{'speaker': 'Tom Robey', 'title': 'IR', 'cont..."
5,2015-03-31,2015-05-01,-0.59,-0.04,-0.55,-1375,pre-market,CHTR,3,2015,2015Q1,20150501T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
6,2015-06-30,2015-08-04,0.06,0.32,-0.26,-81.25,pre-market,CHTR,6,2015,2015Q2,20150804T0000,"[{'speaker': 'Thomas Robey', 'title': 'Senior ..."
7,2015-09-30,2015-10-29,0.53,-0.29,0.82,282.7586,pre-market,CHTR,9,2015,2015Q3,20151029T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
8,2015-12-31,2016-02-04,-1.21,-0.16,-1.05,-656.25,pre-market,CHTR,12,2015,2015Q4,20160204T0000,"[{'speaker': 'Tom Robey', 'title': 'SVP, IR', ..."
9,2016-03-31,2016-04-28,-1.64,-1.47,-0.17,-11.5646,pre-market,CHTR,3,2016,2016Q1,20160428T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."


In [55]:
25*4*12

1200

In [ ]:
df[['av_quarter','transcript']]



,av_quarter,transcript
0,2013Q4,"[{'speaker': 'Ahmed Pasha', 'title': 'VP, IR',..."
1,2014Q1,"[{'speaker': 'Ahmed Pasha', 'title': 'Vice Pre..."
2,2014Q2,"[{'speaker': 'Ahmed Pasha', 'title': 'Vice Pre..."
3,2014Q3,"[{'speaker': 'Ahmed Pasha', 'title': 'Vice Pre..."
4,2014Q4,"[{'speaker': 'Operator', 'title': 'Operator', ..."
5,2015Q1,"[{'speaker': 'Operator', 'title': 'Operator', ..."
6,2015Q2,"[{'speaker': 'Operator', 'title': 'Operator', ..."
7,2015Q3,"[{'speaker': 'Operator', 'title': 'Operator', ..."
8,2015Q4,"[{'speaker': 'Ahmed Pasha', 'title': 'VP, IR',..."
9,2016Q1,"[{'speaker': 'Operator', 'title': 'Operator', ..."


In [ ]:
df = (
    pl.from_pandas(df)
      .explode("transcript")
      .unnest("transcript")
)


TypeError: expected pandas DataFrame or Series, got 'DataFrame'

In [49]:
df

fiscalDateEnding,reportedDate,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,symbol,fiscal_month,fiscal_year,av_quarter,time_from,speaker,title,content,sentiment
datetime[μs],datetime[μs],str,str,str,str,str,str,i8,i32,str,str,str,str,str,str
2013-12-31 00:00:00,2014-02-26 00:00:00,"""0.29""","""0.29""","""0""","""0""","""pre-market""","""AES""",12,2013,"""2013Q4""","""20140226T0000""","""Ahmed Pasha""","""VP, IR""","""Thank you, Alicia. Good mornin…","""0.3"""
2013-12-31 00:00:00,2014-02-26 00:00:00,"""0.29""","""0.29""","""0""","""0""","""pre-market""","""AES""",12,2013,"""2013Q4""","""20140226T0000""","""Andres Gluski""","""CEO""","""Good morning everyone and welc…","""0.7"""
2013-12-31 00:00:00,2014-02-26 00:00:00,"""0.29""","""0.29""","""0""","""0""","""pre-market""","""AES""",12,2013,"""2013Q4""","""20140226T0000""","""Tom O'Flynn""","""CFO""","""Thanks, Andres, good morning e…","""0.5"""
2013-12-31 00:00:00,2014-02-26 00:00:00,"""0.29""","""0.29""","""0""","""0""","""pre-market""","""AES""",12,2013,"""2013Q4""","""20140226T0000""","""Andres Gluski""","""CEO""","""Thanks, Tom. Now turning to Sl…","""0.8"""
2013-12-31 00:00:00,2014-02-26 00:00:00,"""0.29""","""0.29""","""0""","""0""","""pre-market""","""AES""",12,2013,"""2013Q4""","""20140226T0000""","""Operator""","""Operator""","""Thank you. (Operator Instructi…","""0.0"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-09-30 00:00:00,2025-11-05 00:00:00,"""0.89""","""0.64""","""0.25""","""39.0625""","""post-market""","""AES""",9,2025,"""2025Q3""","""20251105T0000""","""Dimple Gosai""","""Analyst""","""More of a housekeeping questio…","""0.1"""
2025-09-30 00:00:00,2025-11-05 00:00:00,"""0.89""","""0.64""","""0.25""","""39.0625""","""post-market""","""AES""",9,2025,"""2025Q3""","""20251105T0000""","""Susan Pasley Harcourt""","""Vice President of Investor Rel…","""We thank everybody for joining…","""0.5"""
2025-09-30 00:00:00,2025-11-05 00:00:00,"""0.89""","""0.64""","""0.25""","""39.0625""","""post-market""","""AES""",9,2025,"""2025Q3""","""20251105T0000""","""Operator""","""Operator""","""This now concludes today's cal…","""0.0"""


In [30]:
s3.ls(f"{bucket}/post-earnings-forecast/transcripts/")

['docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=AES',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=AOS',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=ARE',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=BAX',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=BLDR',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=BXP',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=CAG',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=CPB',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=CRL',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=EPAM',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=FDS',
 'docks-otu-canada-central-data/post-earnings-forecast/transcripts/symbol=FRT',
 'docks-otu-canada-central-data/post-e

In [7]:
q = f"""
SELECT *
FROM read_parquet(
  's3://{bucket}/post-earnings-forecast/transcripts/symbol=GIS/**/*.parquet',
  hive_partitioning = true
);
"""

con.execute(q).df()

,fiscalDateEnding,reportedDate,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,symbol,fiscal_month,fiscal_year,av_quarter,time_from,transcript
0,2014-02-28,2014-03-19,0.62,0.62,0,0,pre-market,GIS,2,2014,2014Q1,20140319T0000,"[{'speaker': 'Kristen Wenker', 'title': 'Senio..."
1,2014-05-31,2014-06-25,0.67,0.72,-0.05,-6.9444,pre-market,GIS,5,2014,2014Q2,20140625T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
2,2014-08-31,2014-09-17,0.61,0.69,-0.08,-11.5942,pre-market,GIS,8,2014,2014Q3,20140917T0000,"[{'speaker': 'Kris Wenker', 'title': 'VP, Inve..."
3,2014-11-30,2014-12-17,0.8,0.76,0.04,5.2632,pre-market,GIS,11,2014,2014Q4,20141217T0000,"[{'speaker': 'Kristen S. Wenker', 'title': 'Se..."
4,2015-02-28,2015-03-18,0.7,0.67,0.03,4.4776,pre-market,GIS,2,2015,2015Q1,20150318T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
5,2015-05-31,2015-07-01,0.75,0.71,0.04,5.6338,pre-market,GIS,5,2015,2015Q2,20150701T0000,"[{'speaker': 'Kris Wenker', 'title': 'Senior V..."
6,2015-08-31,2015-09-22,0.79,0.69,0.1,14.4928,pre-market,GIS,8,2015,2015Q3,20150922T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
7,2015-11-30,2015-12-17,0.82,0.83,-0.01,-1.2048,pre-market,GIS,11,2015,2015Q4,20151217T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
8,2016-02-29,2016-03-23,0.65,0.62,0.03,4.8387,pre-market,GIS,2,2016,2016Q1,20160323T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
9,2016-05-31,2016-06-29,0.66,0.6,0.06,10,pre-market,GIS,5,2016,2016Q2,20160629T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."


In [33]:

s3 = s3fs.S3FileSystem(
    key=os.getenv("S3_ACCESS_KEY"),
    secret=os.getenv("S3_SECRET_KEY"),
    token=os.getenv("AWS_SESSION_TOKEN"),
)

# clear s3 cache to ensure we see latest files
s3.invalidate_cache()
def s3_tree(fs, bucket, prefix="", max_depth=4, indent=""):
    root = f"{bucket}/{prefix}".rstrip("/")
    try:
        items = fs.ls(root, detail=True)
    except FileNotFoundError:
        print(f"{indent}{root}/ (not found)")
        return

    # Split into dirs/files
    dirs = [i for i in items if i.get("type") == "directory"]
    files = [i for i in items if i.get("type") == "file"]

    # Print files with size
    for f in files:
        size = f.get("Size", 0)
        print(f"{indent}- {f['Key'].split('/')[-1]} ({size/1024:.2f} KB)")

    # Recurse into directories
    if max_depth <= 0:
        return
    for d in dirs:
        name = d["Key"].split("/")[-1]
        print(f"{indent}+ {name}/")
        s3_tree(fs, bucket, prefix=d["Key"].split(f"{bucket}/", 1)[-1], max_depth=max_depth-1, indent=indent+"  ")

# Example:
s3_tree(s3, os.getenv("S3_BUCKET"), prefix="post-earnings-forecast", max_depth=5)

+ earnings/
  + symbol=EPAM/
    - 00000000.parquet (5.88 KB)
  + symbol=NVDA/
    - 00000000.parquet (5.85 KB)
  + symbol=NWS/
    - 00000000.parquet (5.69 KB)
  + symbol=TTD/
    - 00000000.parquet (5.47 KB)
  + symbol=WYNN/
    - 00000000.parquet (5.99 KB)
+ snp500/
  - snp500_2026-05-23.csv (25.01 KB)
+ transcripts/
  + symbol=NVDA/
    + av_quarter=2014Q1/
      - 00000000.parquet (23.14 KB)
    + av_quarter=2014Q2/
      - 00000000.parquet (5.49 KB)
    + av_quarter=2014Q3/
      - 00000000.parquet (18.63 KB)
    + av_quarter=2014Q4/
      - 00000000.parquet (23.47 KB)
    + av_quarter=2015Q1/
      - 00000000.parquet (23.28 KB)
    + av_quarter=2015Q2/
      - 00000000.parquet (24.37 KB)
    + av_quarter=2015Q3/
      - 00000000.parquet (22.31 KB)
    + av_quarter=2015Q4/
      - 00000000.parquet (21.51 KB)
    + av_quarter=2016Q1/
      - 00000000.parquet (26.07 KB)
    + av_quarter=2016Q2/
      - 00000000.parquet (26.55 KB)
    + av_quarter=2016Q3/
      - 00000000.parquet (2

In [6]:
# check earnings partition for a symbol
prefix = f"{os.getenv('S3_BUCKET')}/post-earnings-forecast/earnings/"
parts = s3.ls(prefix) 
parts

AttributeError: 'S3' object has no attribute 'ls'

In [5]:
import boto3

s3 = boto3.client('s3')
paginator = s3.get_paginator('list_objects_v2')

total_size = 0
file_count = 0

for page in paginator.paginate(
    Bucket=bucket,
    Prefix='post-earnings-forecast/transcripts/'
):
    for obj in page.get('Contents', []):
        total_size += obj['Size']
        file_count += 1

print(f"Files: {file_count}")
print(f"Size:  {total_size / 1024 / 1024:.2f} MB")

Files: 1286
Size:  28.27 MB


In [7]:
sp500 = pd.read_html(
    'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
)[0][['Symbol', 'GICS Sector']]

HTTPError: HTTP Error 403: Forbidden